# Adım 5: Özellik Mühendisliği
**Kişi 3 sorumluluğu** — `feature/ml-dashboard` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg
from pyspark.sql.window import Window

GOLD_PATH    = './delta_lake/gold'
FEATURE_PATH = './delta_lake/features'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateFeatureEngineering')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[Feature] Gold tablosu okunuyor...')
df = spark.read.format('delta').load(GOLD_PATH)
print(f'[Feature] {df.count():,} kayit yuklendi.')

In [ ]:
# Feature 1: Sicaklik farki (max - min)
df = df.withColumn('temp_range', col('max_temp_c') - col('min_temp_c'))

# Feature 2: Mevsim sayisal kodlama
df = df.withColumn(
    'season_num',
    when(col('season') == 'Winter', 0)
    .when(col('season') == 'Spring', 1)
    .when(col('season') == 'Summer', 2)
    .when(col('season') == 'Autumn', 3)
    .otherwise(0)
)

# Feature 3: Ay (gold katmaninda zaten mevcut)
print('Features 1-3 eklendi: temp_range, season_num, month')
df.select('station_id', 'date', 'avg_temp_c', 'temp_range', 'season_num', 'month').show(5)

## Feature 4-6 ve Kaydetme

In [ ]:
# Feature 4: 7 gunluk hareketli ortalama sicaklik
window_spec = (
    Window.partitionBy('station_id')
    .orderBy('date')
    .rowsBetween(-6, 0)
)
df = df.withColumn('rolling_avg_7', avg('avg_temp_c').over(window_spec))

# Feature 5: Asiri sicaklik bayragi (>35 veya <-20)
df = df.withColumn(
    'is_extreme_temp',
    when((col('avg_temp_c') > 35) | (col('avg_temp_c') < -20), 1).otherwise(0)
)

# Feature 6: Yagis kategorisi (0=Yok, 1=Hafif, 2=Orta, 3=Yogun)
df = df.withColumn(
    'prcp_category',
    when(col('precipitation_mm').isNull() | (col('precipitation_mm') == 0), 0)
    .when(col('precipitation_mm') <= 5, 1)
    .when(col('precipitation_mm') <= 20, 2)
    .otherwise(3)
)

print('Features 4-6 eklendi: rolling_avg_7, is_extreme_temp, prcp_category')

In [ ]:
# Ozellik tablosunu Delta Lake'e yaz
feature_cols = [
    'station_id', 'city_name', 'date', 'year', 'month',
    'avg_temp_c',
    'temp_range', 'season_num', 'rolling_avg_7',
    'is_extreme_temp', 'prcp_category',
    'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min',
]
df_feat = df.select(feature_cols).dropna(subset=['avg_temp_c', 'rolling_avg_7'])
count = df_feat.count()
print(f'[Feature] Kayit sayisi: {count:,}')
df_feat.write.format('delta').mode('overwrite').save(FEATURE_PATH)
print(f'[Feature] Ozellik tablosu yazildi -> {FEATURE_PATH}')
df_feat.show(5, truncate=False)
spark.stop()
print('Ozellik muhendisligi tamamlandi.')